# Module 5 — Numerical Methods for Computational Materials
## Hands-On Python Tutorial

**Level:** IIT M.Tech / PhD Applied Materials / Computational Materials  
**Recommended duration:** 3–4 tutorials of 3 hours each, with independent exercises  
**Prerequisites:** Modules 1–4

---

## Philosophy

Numerical methods are the bridge between mathematical models and computational materials simulations.

The objective is **not** to memorize SciPy functions. Students should understand:

\[
\boxed{\text{Physical model}
\rightarrow
\text{mathematical formulation}
\rightarrow
\text{discretization}
\rightarrow
\text{algorithm}
\rightarrow
\text{Python implementation}
\rightarrow
\text{verification}}
\]

This module therefore emphasizes implementing important algorithms from first principles before using library routines.

### Learning objectives

By the end of this module, students should be able to:

- Estimate derivatives numerically using finite differences.
- Understand truncation and round-off errors.
- Perform numerical integration using Newton–Cotes methods.
- Solve nonlinear equations using bisection and Newton's method.
- Implement interpolation methods.
- Perform numerical optimization.
- Fit parameters using least squares.
- Solve initial-value ODEs.
- Understand the finite-difference method for PDEs.
- Implement the 1D heat equation.
- Implement the 1D diffusion equation.
- Understand stability restrictions for explicit schemes.
- Analyze convergence and discretization error.
- Connect numerical methods to materials problems.
- Understand the basic computational structure of phase-field modelling.
- Use SciPy as a validation/reference implementation rather than a black box.


# 1. Scientific Python setup

We will use:

- NumPy — arrays and vectorized numerical operations
- Matplotlib — visualization
- SciPy — reference numerical algorithms
- Pandas — storing numerical results

The first implementations will generally be written ourselves.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import integrate, optimize, interpolate
from scipy.integrate import solve_ivp
from scipy.optimize import minimize, least_squares

rng = np.random.default_rng(42)

print("Numerical methods environment ready.")


# 2. Numerical error — the foundation

A numerical result can differ from the exact mathematical result for several reasons.

### Truncation error

The mathematical problem is approximated by a finite numerical expression.

### Round-off error

Computers represent numbers with finite precision.

### Discretization error

A continuous domain is represented by a finite set of points.

### Iterative error

An iterative algorithm may stop before reaching the exact solution.

A useful conceptual model is:

\[
E_{\mathrm{total}}
\approx
E_{\mathrm{truncation}}
+
E_{\mathrm{roundoff}}
+
E_{\mathrm{iteration}}.
\]

A good computational scientist learns to **measure and control** these errors.


# 3. Numerical differentiation

Consider

\[
f(x)=\sin x.
\]

The exact derivative is

\[
f'(x)=\cos x.
\]

We will estimate the derivative using finite differences.


In [ ]:
def f(x):
    return np.sin(x)

x = np.linspace(0, 2*np.pi, 200)
exact = np.cos(x)

plt.figure(figsize=(8, 4))
plt.plot(x, exact, label="Exact derivative")
plt.xlabel("x")
plt.ylabel("df/dx")
plt.title("Exact derivative of sin(x)")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## Forward difference

The first-order forward difference is

\[
f'(x)
\approx
\frac{f(x+h)-f(x)}{h}.
\]

Its truncation error is:

\[
O(h).
\]


In [ ]:
def forward_difference(f, x, h):
    return (f(x + h) - f(x)) / h

x0 = 1.0

for h in [1e-1, 1e-2, 1e-3, 1e-4]:
    numerical = forward_difference(f, x0, h)
    exact = np.cos(x0)
    error = abs(numerical - exact)
    print(f"h={h:.0e}, derivative={numerical:.10f}, error={error:.3e}")


## Central difference

The central difference is

\[
f'(x)
\approx
\frac{f(x+h)-f(x-h)}{2h}.
\]

Its truncation error is:

\[
O(h^2).
\]

This is why central differences are often substantially more accurate than forward differences for the same \(h\).


In [ ]:
def central_difference(f, x, h):
    return (f(x + h) - f(x - h)) / (2*h)

for h in [1e-1, 1e-2, 1e-3, 1e-4]:
    numerical = central_difference(f, x0, h)
    exact = np.cos(x0)
    error = abs(numerical - exact)
    print(f"h={h:.0e}, derivative={numerical:.10f}, error={error:.3e}")


# 4. Convergence study

A numerical method should not simply "give an answer".

We should ask:

> How does the error change when the grid spacing \(h\) decreases?

For a second-order method:

\[
E(h)\propto h^2.
\]

Therefore reducing \(h\) by a factor of 2 should reduce the error by approximately a factor of 4, until round-off effects become important.


In [ ]:
hs = 10.0**(-np.arange(1, 7))

forward_errors = []
central_errors = []

for h in hs:
    forward_errors.append(
        abs(forward_difference(f, x0, h) - np.cos(x0))
    )
    central_errors.append(
        abs(central_difference(f, x0, h) - np.cos(x0))
    )

plt.figure(figsize=(7, 5))
plt.loglog(hs, forward_errors, "o-", label="Forward difference")
plt.loglog(hs, central_errors, "s-", label="Central difference")
plt.xlabel("h")
plt.ylabel("Absolute error")
plt.title("Finite-difference convergence")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# Exercise 1 — Numerical differentiation

For

\[
f(x)=e^{-x^2},
\]

implement:

1. forward difference
2. backward difference
3. central difference

Compare them with the exact derivative:

\[
f'(x)=-2xe^{-x^2}.
\]

Construct a log-log convergence plot and estimate the order of accuracy from the slope.


# 5. Second derivative

The central finite-difference approximation for the second derivative is

\[
f''(x)
\approx
\frac{
f(x+h)-2f(x)+f(x-h)
}{h^2}.
\]

For

\[
f(x)=\sin x,
\]

we know

\[
f''(x)=-\sin x.
\]


In [ ]:
def second_derivative(f, x, h):
    return (
        f(x+h) - 2*f(x) + f(x-h)
    ) / h**2

x0 = 1.2

for h in [1e-1, 1e-2, 1e-3, 1e-4]:
    numerical = second_derivative(f, x0, h)
    exact = -np.sin(x0)
    print(
        f"h={h:.0e}, "
        f"numerical={numerical:.10f}, "
        f"error={abs(numerical-exact):.3e}"
    )


# 6. Numerical integration

Many materials models require evaluating integrals such as:

\[
Q=\int_a^b f(x)\,dx.
\]

Examples include:

- accumulated diffusion flux
- probability distributions
- energy functionals
- thermodynamic averages
- volume/area calculations
- normalization constants

We will implement the trapezoidal rule.


## Trapezoidal rule

For equally spaced points:

\[
\int_a^b f(x)\,dx
\approx
h
\left[
\frac{f_0+f_N}{2}
+
\sum_{i=1}^{N-1}f_i
\right].
\]


In [ ]:
def trapezoidal_rule(x, y):
    h = x[1] - x[0]
    return h * (
        0.5*y[0]
        + np.sum(y[1:-1])
        + 0.5*y[-1]
    )

x = np.linspace(0, np.pi, 101)
y = np.sin(x)

I = trapezoidal_rule(x, y)

print("Numerical integral:", I)
print("Exact integral:", 2.0)
print("Error:", abs(I - 2.0))


## Simpson's rule

For equally spaced points and an even number of intervals:

\[
\int_a^b f(x)\,dx
\approx
\frac{h}{3}
\left[
f_0+f_N
+
4\sum f_{\mathrm{odd}}
+
2\sum f_{\mathrm{even}}
\right].
\]

Simpson's rule has fourth-order accuracy for sufficiently smooth functions.


In [ ]:
def simpson_rule(x, y):
    n = len(x) - 1
    if n % 2 != 0:
        raise ValueError("Simpson's rule requires an even number of intervals.")

    h = x[1] - x[0]

    return (
        h/3
        * (
            y[0]
            + y[-1]
            + 4*np.sum(y[1:-1:2])
            + 2*np.sum(y[2:-1:2])
        )
    )

for N in [10, 20, 40, 80]:
    x = np.linspace(0, np.pi, N+1)
    y = np.sin(x)

    trap = trapezoidal_rule(x, y)
    simp = simpson_rule(x, y)

    print(
        f"N={N:3d} | "
        f"trap error={abs(trap-2):.3e} | "
        f"Simpson error={abs(simp-2):.3e}"
    )


# 7. Compare with SciPy

After implementing a method ourselves, we can compare with a trusted library.

This is an important scientific-computing habit:

\[
\boxed{\text{Implement} \rightarrow \text{verify} \rightarrow \text{use library}}
\]

not:

\[
\boxed{\text{call function} \rightarrow \text{assume correct}}
\]


In [ ]:
x = np.linspace(0, np.pi, 1001)
y = np.sin(x)

manual = trapezoidal_rule(x, y)
scipy_value = integrate.trapezoid(y, x)

print("Manual:", manual)
print("SciPy :", scipy_value)
print("Difference:", abs(manual - scipy_value))


# 8. Root finding

A root of

\[
f(x)=0
\]

is important in many materials problems:

- equilibrium conditions
- phase-transition calculations
- constitutive equations
- nonlinear material models
- implicit equations of state

We will solve

\[
f(x)=x^3-x-2=0.
\]


In [ ]:
def root_function(x):
    return x**3 - x - 2

xx = np.linspace(-2, 2, 400)

plt.figure(figsize=(7, 4))
plt.plot(xx, root_function(xx))
plt.axhline(0, linestyle="--")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Root-finding problem")
plt.grid(alpha=0.25)
plt.show()


# 9. Bisection method

If

\[
f(a)f(b)<0,
\]

then, assuming continuity, at least one root lies inside \([a,b]\).

Bisection repeatedly halves the interval.

Algorithm:

1. calculate midpoint
2. determine which half contains the sign change
3. replace the corresponding endpoint
4. repeat

The interval width decreases as:

\[
\frac{b-a}{2^n}.
\]


In [ ]:
def bisection(f, a, b, tol=1e-10, max_iter=100):
    fa = f(a)
    fb = f(b)

    if fa * fb > 0:
        raise ValueError("Root is not bracketed.")

    history = []

    for iteration in range(max_iter):
        c = 0.5*(a+b)
        fc = f(c)

        history.append((iteration, a, b, c, fc))

        if abs(fc) < tol or 0.5*(b-a) < tol:
            return c, pd.DataFrame(
                history,
                columns=["iteration", "a", "b", "midpoint", "f(midpoint)"]
            )

        if fa * fc < 0:
            b = c
            fb = fc
        else:
            a = c
            fa = fc

    return c, pd.DataFrame(
        history,
        columns=["iteration", "a", "b", "midpoint", "f(midpoint)"]
    )

root, history = bisection(root_function, 1, 2)

print("Root:", root)
print("Iterations:", len(history))
history.head()


# 10. Newton–Raphson method

Newton's method uses:

\[
x_{n+1}
=
x_n
-
\frac{f(x_n)}{f'(x_n)}.
\]

For

\[
f(x)=x^3-x-2,
\]

\[
f'(x)=3x^2-1.
\]

Newton's method can converge very rapidly, but it can fail if the initial guess is poor or if the derivative becomes small.


In [ ]:
def newton_method(f, df, x0, tol=1e-10, max_iter=50):
    x = x0
    history = []

    for iteration in range(max_iter):
        fx = f(x)
        dfx = df(x)

        history.append((iteration, x, fx))

        if abs(fx) < tol:
            break

        if abs(dfx) < 1e-14:
            raise ZeroDivisionError("Derivative too small.")

        x = x - fx/dfx

    return x, pd.DataFrame(
        history,
        columns=["iteration", "x", "f(x)"]
    )

def droot_function(x):
    return 3*x**2 - 1

root_newton, newton_history = newton_method(
    root_function,
    droot_function,
    1.5
)

print("Root:", root_newton)
newton_history


# Exercise 2 — Root finding

Solve:

\[
\cos x-x=0
\]

using:

1. bisection
2. Newton's method
3. `scipy.optimize.brentq`
4. `scipy.optimize.newton`

Compare:

- number of iterations
- initial-condition requirements
- robustness
- accuracy

Explain why a bracketing method is generally safer than Newton's method when only a sign-changing interval is known.


# 11. Interpolation

Suppose a materials experiment produces measurements only at discrete temperatures:

\[
(T_i,k_i).
\]

We may need an estimate between measured points.

Interpolation approximates:

\[
f(x^\*)\quad
\text{for}\quad
x_i<x^\*<x_{i+1}.
\]

We will compare linear and cubic interpolation.


In [ ]:
T = np.array([300, 400, 500, 600, 700])
k = np.array([14.8, 16.2, 18.1, 20.5, 23.4])

T_dense = np.linspace(300, 700, 300)

linear = interpolate.interp1d(
    T, k, kind="linear"
)

cubic = interpolate.interp1d(
    T, k, kind="cubic"
)

plt.figure(figsize=(8, 5))
plt.scatter(T, k, label="Experimental points")
plt.plot(T_dense, linear(T_dense), label="Linear")
plt.plot(T_dense, cubic(T_dense), label="Cubic")
plt.xlabel("Temperature (K)")
plt.ylabel("Thermal conductivity")
plt.title("Interpolation of materials data")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


### Important warning

Interpolation is not the same as extrapolation.

Inside the measured domain, interpolation can be reasonable.

Outside the measured domain, predictions may become physically meaningless.

This distinction is especially important in materials databases with sparse measurements.


# 12. Optimization

Optimization appears throughout computational materials:

- minimizing free energy
- fitting material parameters
- finding equilibrium configurations
- calibrating constitutive models
- optimizing processing conditions
- inverse problems

Consider:

\[
F(x)=
(x-2)^2+0.5\sin(5x).
\]

We want to find a minimum.


In [ ]:
def energy(x):
    return (x - 2)**2 + 0.5*np.sin(5*x)

xx = np.linspace(-1, 5, 1000)

plt.figure(figsize=(8, 4))
plt.plot(xx, energy(xx))
plt.xlabel("x")
plt.ylabel("F(x)")
plt.title("Example energy landscape")
plt.grid(alpha=0.25)
plt.show()


In [ ]:
result = minimize(
    lambda z: energy(z[0]),
    x0=[3.5]
)

print("Minimum:", result.x)
print("Energy:", result.fun)
print("Success:", result.success)


# 13. Optimization and local minima

The energy landscape can contain several local minima.

Therefore:

- the starting point can matter
- local optimization may not find the global minimum
- physical constraints may need to be included

In materials science this is analogous to energy landscapes containing metastable states.


In [ ]:
starts = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5]

for start in starts:
    result = minimize(
        lambda z: energy(z[0]),
        x0=[start]
    )
    print(
        f"start={start:4.1f} -> "
        f"minimum={result.x[0]:.5f}, "
        f"F={result.fun:.5f}"
    )


# 14. Least-squares parameter fitting

Suppose a material follows a simple constitutive model:

\[
\sigma =
\sigma_0 + K\epsilon.
\]

Experimental measurements contain noise.

We can estimate \(\sigma_0\) and \(K\) by minimizing:

\[
J(\theta)=
\sum_i
\left[
\sigma_i-\hat{\sigma}(\epsilon_i;\theta)
\right]^2.
\]


In [ ]:
strain = np.linspace(0, 0.02, 40)

true_sigma0 = 250
true_K = 8000

stress = (
    true_sigma0
    + true_K*strain
    + rng.normal(0, 25, len(strain))
)

A = np.column_stack([
    np.ones_like(strain),
    strain
])

params, *_ = np.linalg.lstsq(A, stress, rcond=None)

sigma0_fit, K_fit = params

print("Fitted sigma0:", sigma0_fit)
print("Fitted K:", K_fit)


In [ ]:
stress_fit = sigma0_fit + K_fit*strain

plt.figure(figsize=(8, 5))
plt.scatter(strain, stress, label="Synthetic experiment")
plt.plot(strain, stress_fit, label="Least-squares fit")
plt.xlabel("Strain")
plt.ylabel("Stress (MPa)")
plt.title("Constitutive parameter fitting")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# 15. Ordinary differential equations

Many materials processes are described by ODEs.

Examples:

- reaction kinetics
- cooling curves
- sintering models
- population of defects
- phase transformations
- lumped thermal models

Consider Newtonian cooling:

\[
\frac{dT}{dt}
=
-k(T-T_\mathrm{env}).
\]

The analytical solution is:

\[
T(t)=T_\mathrm{env}
+
(T_0-T_\mathrm{env})e^{-kt}.
\]


In [ ]:
T_env = 300.0
T0 = 1000.0
k = 0.15

def cooling_rhs(t, T):
    return -k*(T - T_env)

t = np.linspace(0, 30, 200)

T_exact = (
    T_env
    + (T0 - T_env)*np.exp(-k*t)
)

plt.figure(figsize=(8, 5))
plt.plot(t, T_exact, label="Analytical")
plt.xlabel("Time")
plt.ylabel("Temperature (K)")
plt.title("Newtonian cooling")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# 16. Euler method for an ODE

For

\[
\frac{dT}{dt}=f(t,T),
\]

the explicit Euler method is:

\[
T_{n+1}
=
T_n+\Delta t\,f(t_n,T_n).
\]

This is the simplest example of time discretization.


In [ ]:
def euler_ode(rhs, t, y0):
    y = np.zeros((len(t), len(np.atleast_1d(y0))))
    y[0] = np.atleast_1d(y0)

    for n in range(len(t)-1):
        dt = t[n+1] - t[n]
        y[n+1] = y[n] + dt*rhs(t[n], y[n])

    return y

for dt in [1.0, 0.2, 0.05]:
    t_euler = np.arange(0, 30 + dt, dt)

    y = euler_ode(
        cooling_rhs,
        t_euler,
        [T0]
    )[:, 0]

    error = abs(y[-1] - T_exact[-1])

    print(
        f"dt={dt:.3f}, "
        f"final temperature={y[-1]:.3f}, "
        f"error={error:.3e}"
    )


# 17. SciPy ODE solver

For production work, we can use `solve_ivp`.

The point of first implementing Euler is to understand what the solver is doing conceptually.


In [ ]:
solution = solve_ivp(
    cooling_rhs,
    (0, 30),
    [T0],
    t_eval=t
)

plt.figure(figsize=(8, 5))
plt.plot(t, T_exact, label="Analytical")
plt.plot(
    solution.t,
    solution.y[0],
    "--",
    label="solve_ivp"
)
plt.xlabel("Time")
plt.ylabel("Temperature (K)")
plt.title("ODE solution")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# 18. Finite Difference Method for PDEs

We now move from ODEs to PDEs.

Consider the 1D heat equation:

\[
\frac{\partial T}{\partial t}
=
\alpha
\frac{\partial^2T}{\partial x^2}.
\]

Here:

- \(T(x,t)\) is temperature
- \(\alpha\) is thermal diffusivity

This equation is central to heat transfer and appears in many computational-materials problems.


## Spatial discretization

At grid point \(x_i\):

\[
\frac{\partial^2T}{\partial x^2}
\approx
\frac{
T_{i+1}-2T_i+T_{i-1}
}{\Delta x^2}.
\]

For explicit time integration:

\[
T_i^{n+1}
=
T_i^n
+
r
\left(
T_{i+1}^n
-2T_i^n
+T_{i-1}^n
\right),
\]

where

\[
r=
\frac{\alpha\Delta t}{\Delta x^2}.
\]


# 19. Stability of the explicit heat-equation scheme

For the standard 1D explicit scheme, stability requires approximately:

\[
\boxed{
r\leq \frac12
}
\]

or

\[
\boxed{
\Delta t
\leq
\frac{\Delta x^2}{2\alpha}.
}
\]

This is a crucial computational-materials concept:

**the time step is constrained by the spatial resolution.**

If we make the grid twice as fine:

\[
\Delta x\rightarrow\frac{\Delta x}{2},
\]

then the maximum stable time step becomes approximately:

\[
\Delta t_{\max}
\rightarrow
\frac{\Delta t_{\max}}{4}.
\]

This can make explicit simulations computationally expensive.


# 20. Implement the 1D heat equation

We consider a slab:

\[
0\le x\le L.
\]

Boundary conditions:

\[
T(0,t)=T_\mathrm{left},
\]

\[
T(L,t)=T_\mathrm{right}.
\]

Initial condition:

\[
T(x,0)=T_\mathrm{initial}(x).
\]


In [ ]:
def heat_equation_explicit(
    L=1.0,
    alpha=0.01,
    nx=101,
    dt=0.0001,
    total_time=0.1,
    T_left=100.0,
    T_right=0.0
):
    dx = L/(nx-1)
    r = alpha*dt/dx**2

    if r > 0.5:
        raise ValueError(
            f"Unstable explicit scheme: r={r:.3f} > 0.5"
        )

    nt = int(total_time/dt) + 1

    x = np.linspace(0, L, nx)
    T = np.zeros(nx)

    # Hot left boundary, cold right boundary
    T[:] = 0.0
    T[0] = T_left
    T[-1] = T_right

    snapshots = [T.copy()]
    snapshot_times = [0.0]

    for n in range(1, nt):
        T_old = T.copy()

        T[1:-1] = (
            T_old[1:-1]
            + r*(
                T_old[2:]
                - 2*T_old[1:-1]
                + T_old[:-2]
            )
        )

        T[0] = T_left
        T[-1] = T_right

        if n % max(1, nt//5) == 0:
            snapshots.append(T.copy())
            snapshot_times.append(n*dt)

    return x, np.array(snapshot_times), np.array(snapshots), r

x, times, snapshots, r = heat_equation_explicit()

print("Stability parameter r =", r)


In [ ]:
plt.figure(figsize=(8, 5))

for t_i, T_i in zip(times, snapshots):
    plt.plot(x, T_i, label=f"t={t_i:.3f}")

plt.xlabel("x")
plt.ylabel("Temperature")
plt.title("1D heat equation — explicit FDM")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# 21. Visualize the full space-time solution

A heat-equation simulation produces a field:

\[
T=T(x,t).
\]

A heatmap is useful for visualizing the evolution of the field.


In [ ]:
# A longer simulation with regularly stored snapshots
L = 1.0
alpha = 0.01
nx = 101
dx = L/(nx-1)

dt = 0.4*dx**2/alpha
total_time = 1.0

x, times, snapshots, r = heat_equation_explicit(
    L=L,
    alpha=alpha,
    nx=nx,
    dt=dt,
    total_time=total_time
)

print("r =", r)
print("Snapshots shape:", snapshots.shape)

plt.figure(figsize=(9, 5))
plt.imshow(
    snapshots,
    aspect="auto",
    origin="lower",
    extent=[0, L, times[0], times[-1]]
)
plt.xlabel("x")
plt.ylabel("Time")
plt.title("Temperature evolution")
plt.colorbar(label="Temperature")
plt.show()


# 22. Steady-state analytical solution

At steady state:

\[
\frac{\partial T}{\partial t}=0.
\]

Therefore:

\[
\frac{d^2T}{dx^2}=0.
\]

The solution is linear:

\[
T(x)=
T_\mathrm{left}
+
\frac{T_\mathrm{right}-T_\mathrm{left}}{L}x.
\]

We can verify that the numerical solution approaches this solution.


In [ ]:
T_left = 100
T_right = 0

T_steady = (
    T_left
    + (T_right - T_left)*x/L
)

plt.figure(figsize=(8, 5))
plt.plot(x, T_steady, "k--", label="Analytical steady state")
plt.plot(x, snapshots[-1], label="Numerical")
plt.xlabel("x")
plt.ylabel("Temperature")
plt.title("Steady-state verification")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# Exercise 3 — Heat equation

Modify the solver to model a material slab with:

- \(L=5\) mm
- \(\alpha=1.2\times10^{-5}\,\mathrm{m^2/s}\)
- left boundary = 800 K
- right boundary = 300 K

Tasks:

1. Determine a stable time step.
2. Run the simulation.
3. Plot temperature profiles.
4. Plot a space-time heatmap.
5. Compare the final state with the analytical steady-state solution.
6. Repeat with two spatial resolutions.
7. Discuss computational cost versus accuracy.


# 23. Diffusion equation

The concentration field \(C(x,t)\) obeys:

\[
\frac{\partial C}{\partial t}
=
D
\frac{\partial^2C}{\partial x^2}.
\]

This has the same mathematical structure as the heat equation.

Therefore, the same finite-difference machinery can model:

- atomic diffusion
- dopant diffusion
- vacancy transport
- concentration homogenization
- interdiffusion


# 24. Implement 1D diffusion

We begin with a localized concentration distribution and allow it to spread.


In [ ]:
def diffusion_explicit(
    L=1.0,
    D=0.01,
    nx=201,
    dt=0.00005,
    total_time=0.02
):
    dx = L/(nx-1)
    r = D*dt/dx**2

    if r > 0.5:
        raise ValueError(
            f"Unstable explicit scheme: r={r:.3f} > 0.5"
        )

    nt = int(total_time/dt) + 1
    x = np.linspace(0, L, nx)

    C = np.exp(
        -((x - 0.5*L)/(0.05*L))**2
    )

    snapshots = [C.copy()]
    times = [0.0]

    for n in range(1, nt):
        C_old = C.copy()

        C[1:-1] = (
            C_old[1:-1]
            + r*(
                C_old[2:]
                - 2*C_old[1:-1]
                + C_old[:-2]
            )
        )

        # Zero-gradient boundary approximation
        C[0] = C[1]
        C[-1] = C[-2]

        if n % max(1, nt//5) == 0:
            snapshots.append(C.copy())
            times.append(n*dt)

    return x, np.array(times), np.array(snapshots), r

x_d, times_d, C_snapshots, r_d = diffusion_explicit()

print("Diffusion stability parameter:", r_d)


In [ ]:
plt.figure(figsize=(8, 5))

for t_i, C_i in zip(times_d, C_snapshots):
    plt.plot(x_d, C_i, label=f"t={t_i:.4f}")

plt.xlabel("x")
plt.ylabel("Concentration")
plt.title("1D diffusion")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# 25. Conservation check

For diffusion with zero-flux boundaries, the total amount of species should remain approximately constant:

\[
M(t)=\int_0^L C(x,t)\,dx.
\]

This is an example of a **physical verification test**.


In [ ]:
mass = [
    np.trapezoid(C_i, x_d)
    for C_i in C_snapshots
]

plt.figure(figsize=(7, 4))
plt.plot(times_d, mass, "o-")
plt.xlabel("Time")
plt.ylabel("Integrated concentration")
plt.title("Mass conservation check")
plt.grid(alpha=0.25)
plt.show()

print("Initial mass:", mass[0])
print("Final mass:", mass[-1])
print("Relative change:", abs(mass[-1]-mass[0])/abs(mass[0]))


# 26. Numerical verification: grid refinement

A powerful test is **grid convergence**.

Run the same physical problem with:

- 51 grid points
- 101 grid points
- 201 grid points

Compare a quantity of interest.

If the solution changes substantially with grid resolution, the calculation is not yet grid converged.


In [ ]:
def heat_final_profile(nx, alpha=0.01, L=1.0):
    dx = L/(nx-1)
    dt = 0.4*dx**2/alpha

    x, times, snapshots, r = heat_equation_explicit(
        L=L,
        alpha=alpha,
        nx=nx,
        dt=dt,
        total_time=1.0
    )

    return x, snapshots[-1]

for nx in [51, 101, 201]:
    x_ref, T_ref = heat_final_profile(nx)
    print(
        f"nx={nx:3d}, "
        f"center temperature={T_ref[len(T_ref)//2]:.6f}"
    )


# 27. CFL-like stability thinking

Many numerical PDE methods have a dimensionless stability parameter.

For the explicit diffusion/heat equation:

\[
r=\frac{\alpha\Delta t}{\Delta x^2}.
\]

The numerical scientist should think in terms of **dimensionless groups** rather than arbitrary code parameters.

This is analogous to physical dimensionless numbers such as:

- Fourier number
- Peclet number
- Reynolds number
- Courant number

Numerical stability is often controlled by such ratios.


# 28. A simple phase-field model

Phase-field modelling introduces a continuous order parameter:

\[
\phi(\mathbf{x},t).
\]

A simplified Allen–Cahn-type equation is:

\[
\frac{\partial\phi}{\partial t}
=
-M
\frac{\delta F}{\delta\phi}.
\]

For a simple free-energy functional:

\[
F=
\int
\left[
f(\phi)
+
\frac{\kappa}{2}|\nabla\phi|^2
\right]dV,
\]

with double-well potential:

\[
f(\phi)
=
\frac14(\phi^2-1)^2,
\]

the evolution can be approximated by:

\[
\frac{\partial\phi}{\partial t}
=
-M
\left[
\phi^3-\phi
-
\kappa\nabla^2\phi
\right].
\]

This provides a natural bridge from numerical PDEs to microstructure evolution.


# 29. 1D Allen–Cahn demonstration

We implement a simple explicit scheme.

This is **not a production phase-field code**. It is an educational demonstration of how:

- a free-energy derivative
- a spatial derivative
- time integration
- numerical stability

come together.


In [ ]:
def allen_cahn_1d(
    nx=201,
    L=1.0,
    M=1.0,
    kappa=0.002,
    dt=1e-5,
    total_time=0.01
):
    dx = L/(nx-1)

    x = np.linspace(0, L, nx)

    # Small random perturbation around phi = 0
    phi = 0.05*rng.normal(size=nx)

    nt = int(total_time/dt) + 1

    snapshots = [phi.copy()]
    times = [0.0]

    for n in range(1, nt):
        old = phi.copy()

        laplacian = np.zeros_like(phi)
        laplacian[1:-1] = (
            old[2:]
            - 2*old[1:-1]
            + old[:-2]
        ) / dx**2

        dF_dphi = (
            old**3
            - old
            - kappa*laplacian
        )

        phi[1:-1] = (
            old[1:-1]
            - M*dt*dF_dphi[1:-1]
        )

        # Neumann-type zero-gradient boundaries
        phi[0] = phi[1]
        phi[-1] = phi[-2]

        if n % max(1, nt//5) == 0:
            snapshots.append(phi.copy())
            times.append(n*dt)

    return x, np.array(times), np.array(snapshots)

x_phi, t_phi, phi_snapshots = allen_cahn_1d()

print(phi_snapshots.shape)


In [ ]:
plt.figure(figsize=(8, 5))

for t_i, phi_i in zip(t_phi, phi_snapshots):
    plt.plot(x_phi, phi_i, label=f"t={t_i:.4f}")

plt.xlabel("x")
plt.ylabel("Order parameter φ")
plt.title("Educational 1D Allen–Cahn simulation")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# 30. What the phase-field code teaches

Even this small example contains the architecture of much larger computational materials codes:

\[
\boxed{
\text{Free energy}
\rightarrow
\text{functional derivative}
\rightarrow
\text{spatial discretization}
\rightarrow
\text{time integration}
\rightarrow
\text{microstructure evolution}
}
\]

Real phase-field models may include:

- multiple order parameters
- concentration fields
- anisotropic interfacial energy
- elasticity
- coupled thermal fields
- multiple phases
- conserved dynamics (Cahn–Hilliard)
- adaptive meshes
- implicit or semi-implicit time integration


# 31. Numerical differentiation of simulation fields

Once a simulation produces a field \(T(x)\), \(C(x)\), or \(\phi(x)\), the same finite-difference methods can calculate:

\[
\frac{\partial T}{\partial x},
\qquad
\frac{\partial^2T}{\partial x^2}.
\]

For example, the heat flux follows Fourier's law:

\[
q=-k\frac{\partial T}{\partial x}.
\]

This demonstrates how numerical differentiation becomes a physical quantity.


In [ ]:
# Use the final heat-equation profile
dTdx = np.gradient(snapshots[-1], x)

k_thermal = 20.0
q = -k_thermal*dTdx

fig = plt.figure(figsize=(8, 4))
plt.plot(x, q)
plt.xlabel("x")
plt.ylabel("Heat flux")
plt.title("Heat flux calculated from numerical temperature gradient")
plt.grid(alpha=0.25)
plt.show()


# 32. Putting numerical methods together

A realistic computational-materials workflow may look like:

### Example: thermal treatment

\[
\text{temperature history}
\rightarrow
\text{ODE/PDE}
\rightarrow
T(x,t)
\]

then:

\[
T(x,t)
\rightarrow
\nabla T
\rightarrow
q(x,t)
\]

then:

\[
T(x,t),q(x,t)
\rightarrow
\text{microstructure model}
\rightarrow
\text{properties}.
\]

The important point is that numerical methods are **building blocks**, not isolated programming exercises.


# 33. Numerical-method selection guide

| Problem | Typical method |
|---|---|
| Derivative | Finite difference |
| Integral | Trapezoidal / Simpson / Gaussian quadrature |
| Nonlinear equation | Bisection / Newton / Brent |
| Missing intermediate value | Interpolation |
| Parameter estimation | Least squares |
| Minimum of a function | Optimization |
| Initial-value ODE | Euler / RK / solve_ivp |
| 1D diffusion | Finite difference |
| Heat equation | Finite difference / FEM / FVM |
| General PDE | FDM / FEM / FVM |
| Phase-field evolution | PDE discretization + time integration |

The appropriate method depends on accuracy, stability, geometry, boundary conditions, computational cost, and physical conservation requirements.


# 34. Verification versus validation

These terms are fundamental.

### Verification

> Did we solve the mathematical equations correctly?

Examples:

- compare with an analytical solution
- grid refinement
- time-step refinement
- convergence rate
- conservation checks

### Validation

> Do the mathematical equations represent the real physical system?

Examples:

- compare simulation with experimental temperature profiles
- compare predicted diffusion coefficient with measured data
- compare phase evolution with microscopy

A code can be verified but still fail validation if the physical model is inadequate.


# Exercise 4 — Verification study

For the heat equation:

1. choose three spatial resolutions
2. choose stable time steps
3. calculate the numerical solution
4. compare against the analytical steady state
5. calculate an \(L_2\) error:

\[
E_2=
\sqrt{
\frac{1}{N}
\sum_i
(T_i-T_i^{exact})^2
}
\]

6. plot error versus grid spacing
7. estimate the observed convergence rate


# 35. Advanced exercise — diffusion length scale

Diffusion has a characteristic length scale:

\[
\ell_D\sim\sqrt{Dt}.
\]

For a material with:

\[
D=10^{-14}\,\mathrm{m^2/s},
\]

estimate the diffusion length after:

- 1 s
- 1 hour
- 1 day
- 1 week

Then design a numerical grid that can resolve the diffusion profile.

Discuss why physical length scales should guide computational-domain and grid selection.


In [ ]:
D = 1e-14

times_seconds = np.array([
    1,
    3600,
    24*3600,
    7*24*3600
])

diffusion_lengths = np.sqrt(D*times_seconds)

pd.DataFrame({
    "time_s": times_seconds,
    "diffusion_length_m": diffusion_lengths,
    "diffusion_length_um": diffusion_lengths*1e6
})


# 36. Advanced exercise — compare Euler and RK methods

Solve:

\[
\frac{dy}{dt}=-\lambda y
\]

using:

1. Euler
2. `solve_ivp`

Compare the numerical solution with:

\[
y(t)=y_0e^{-\lambda t}.
\]

Repeat for progressively larger time steps.

Discuss:

- accuracy
- stability
- computational cost
- why higher-order methods can permit larger time steps


# 37. Mini-project — 1D thermal processing simulation

## Problem

Model a material slab undergoing a thermal cycle.

The surface temperature is prescribed as:

1. heat from 300 K to 1000 K
2. hold at 1000 K
3. cool back to 300 K

Use the 1D heat equation:

\[
\frac{\partial T}{\partial t}
=
\alpha
\frac{\partial^2T}{\partial x^2}.
\]

### Requirements

1. Implement the spatial discretization yourself.
2. Use an explicit time integrator.
3. Determine the stability limit.
4. Apply time-dependent boundary conditions.
5. Plot \(T(x,t)\).
6. Calculate thermal gradients.
7. Calculate heat flux.
8. Perform a grid-refinement study.
9. Perform a time-step study.
10. Discuss computational cost.

### Extension

Use the resulting thermal history as an input to a simple temperature-dependent diffusion model.


# 38. Mini-project — diffusion couple

Model two materials initially separated by an interface:

\[
C(x,0)=
\begin{cases}
C_A,&x<L/2\\
C_B,&x>L/2
\end{cases}
\]

and solve:

\[
\frac{\partial C}{\partial t}
=
D\frac{\partial^2C}{\partial x^2}.
\]

### Questions

- How does the interface broaden?
- How does the characteristic length scale change with time?
- Is total solute conserved?
- What happens when \(D\) is increased by a factor of 10?
- What happens when the grid is refined?
- What stability restriction controls the time step?


# 39. Mini-project — introductory phase-field model

Implement a 1D Allen–Cahn model.

Investigate:

- effect of mobility \(M\)
- effect of gradient-energy coefficient \(\kappa\)
- effect of grid spacing
- effect of time step
- evolution of the order parameter
- change in free energy

### Suggested scientific question

> How does the characteristic interface width depend on the gradient-energy coefficient?

Do not merely generate plots. Form a hypothesis, perform simulations, and interpret the numerical results physically.


# 40. Recommended coding practice

For serious numerical work, separate the code into:

```text
1. Physical parameters
2. Numerical parameters
3. Initial conditions
4. Boundary conditions
5. Solver
6. Diagnostics
7. Visualization
8. Verification
```

This makes it easier to modify a physical model without accidentally changing the numerical algorithm.


# 41. Recommended numerical-software libraries

### Core
- NumPy
- SciPy
- Matplotlib

### PDE / scientific computing extensions
- FiPy
- FEniCS / FEniCSx
- SfePy
- Dedalus

### Materials simulation
- ASE
- pymatgen

### Phase-field
- FiPy is particularly useful for teaching finite-volume PDE methods.
- MOOSE is widely used for multiphysics and phase-field research workflows.

Students should first understand the numerical method before relying on a sophisticated solver framework.


# 42. Numerical methods and later modules

This module provides the computational foundation for:

### Module 6
Data preparation and feature engineering

### Module 7
Classical machine learning

### Module 8
Deep learning

### Module 9
Materials informatics

### Capstone
Integrated computational materials modelling

The connection is:

\[
\boxed{
\text{Physics}
\rightarrow
\text{Numerical simulation}
\rightarrow
\text{Data}
\rightarrow
\text{Features}
\rightarrow
\text{ML}
}
\]

Students should eventually be able to generate simulation data themselves and then use machine learning to analyze, accelerate, or replace expensive simulations.


# 43. Final assessment

## Part A — Numerical methods

Implement from scratch:

- one differentiation method
- one integration method
- one root-finding method
- one ODE solver

## Part B — PDE

Implement the 1D heat or diffusion equation using finite differences.

## Part C — Verification

Demonstrate:

- grid convergence
- time-step convergence
- stability
- comparison with an analytical or reference solution
- conservation where applicable

## Part D — Materials interpretation

Explain the numerical solution in terms of a real materials-science process.

## Part E — Scientific report

Include:

1. governing equations
2. discretization
3. algorithm
4. numerical parameters
5. results
6. convergence tests
7. physical interpretation
8. limitations


# 44. Key takeaways

The most important ideas are:

\[
\boxed{\text{Discretization turns equations into algorithms}}
\]

\[
\boxed{\text{Accuracy and stability are different concepts}}
\]

\[
\boxed{\text{Smaller } \Delta x \text{ does not automatically mean a better simulation}}
\]

\[
\boxed{\text{Time step and spatial resolution can be coupled}}
\]

\[
\boxed{\text{Numerical solutions must be verified}}
\]

\[
\boxed{\text{Physical conservation laws are powerful diagnostics}}
\]

\[
\boxed{\text{Libraries should support understanding, not replace it}}
\]

The ultimate goal is to make students capable of reading a mathematical model from a computational-materials paper and turning it into a **working, verifiable Python simulation**.
